### Comparaison finale sur ASVspoof2019-LA eval

L'objectif de ce dernier step est de comparer les deux modèles finaux trouver dans nos step2 respectifs :  
- SVM optimisé (pipeline Nora)  
- Random Forest optimisé (pipeline Maryam)

L’objectif est de vérifier la robustesse en conditions réelles sur des spoof non labéllisé.  
Un meilleur ROC-AUC et un F1-score élevé indiquent une meilleure capacité à différencier les audio bonafide des audios spoof.

On va comparer les métriques pour déterminer lequel des deux pipelines généralise le mieux sur le jeu de test final.


In [18]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import joblib

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

SRC_DIR = Path("./src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_utils import load_asvspoof_protocol, attach_paths_by_utt

print("Utils loaded.")


Utils loaded.


In [19]:
model_svm = joblib.load(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\data\model_svm.pkl")
model_rf  = joblib.load(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\data\model_rf.pkl")

print("Modèles chargés :")
print(" - SVM :", model_svm)
print(" - Random Forest :", model_rf)


Modèles chargés :
 - SVM : Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc',
                 SVC(C=1, class_weight='balanced', probability=True,
                     random_state=42))])
 - Random Forest : RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)


C:\Users\norab\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\norab\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [20]:
DATA_ROOT = Path(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\archive\LA")

PROTO_EVAL = DATA_ROOT / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.eval.trl.txt"
AUDIO_ROOT_EVAL = DATA_ROOT / "ASVspoof2019_LA_eval" / "flac"


In [21]:
eval_df = attach_paths_by_utt(load_asvspoof_protocol(PROTO_EVAL), AUDIO_ROOT_EVAL)

print("Eval size:", eval_df.shape)
eval_df.head()


Eval size: (71237, 3)


,utt,is_spoof,path
0,LA_E_2834763,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
1,LA_E_8877452,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
2,LA_E_6828287,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
3,LA_E_6977360,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
4,LA_E_5932896,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...


In [22]:
def extract_features(path, sr=16000, n_mfcc=13):
    y, _ = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

# On test sur 5000 échantillons pris au hasard car tous les fichiers eval seraient trop nombreux
eval_sample = eval_df.sample(5000, random_state=42)

X_eval = np.vstack([extract_features(p) for p in eval_sample["path"]])
y_eval = eval_sample["is_spoof"].to_numpy()

X_eval.shape, y_eval.shape


((5000, 26), (5000,))

In [23]:
def evaluate(name, model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y, y_pred))
    print("F1-score :", f1_score(y, y_pred))
    print("ROC-AUC  :", roc_auc_score(y, y_proba))
    print("Confusion matrix:\n", confusion_matrix(y, y_pred))

evaluate("SVM (final)", model_svm, X_eval, y_eval)
evaluate("Random Forest (final)", model_rf, X_eval, y_eval)



===== SVM (final) =====
Accuracy : 0.8798
F1-score : 0.9328566640598815
ROC-AUC  : 0.886613059811743
Confusion matrix:
 [[ 224  306]
 [ 295 4175]]

===== Random Forest (final) =====
Accuracy : 0.884
F1-score : 0.9370386452453322
ROC-AUC  : 0.8658399814275464
Confusion matrix:
 [[ 104  426]
 [ 154 4316]]


### **Analyse des résultats — SVM**

- Accuracy : 0.8798  
- F1-score : 0.9329  
- ROC-AUC : 0.8866

Matrice de confusion :

              Prédit bonafide   Prédit spoof 
Réel bonafide    ((224)            (306)
Réel spoof        (295)            (4175))

**Interprétation :**
Le SVM conserve de très bonnes performances globales. Il détecte correctement la quasi-totalité des attaques spoof (4175 sur 4470), ce qui explique son F1-score élevé.  
Cependant, il reste plus sensible aux faux positifs que le Random Forest, ce qui se traduit par 306 bonafide mal classés.  
Ce comportement est cohérent avec la structure initialement déséquilibrée des données LA.

---

### **Analyse des résultats — Random Forest**

- Accuracy : 0.8840  
- F1-score : 0.9370  
- ROC-AUC : 0.8658

Matrice de confusion :

              Prédit bonafide   Prédit spoof 
Réel bonafide    ((104)            (426)
Réel spoof        (154)            (4316))

**Interprétation :**  
Le Random Forest obtient les meilleures performances globales, notamment grâce à une réduction notable des faux positifs (seulement 104 bonafide classés en spoof).  
Son F1-score est le meilleur des deux modèles, ce qui en fait une option plus robuste pour une utilisation réelle.  
Son ROC-AUC est légèrement inférieur à celui du SVM, ce qui indique une séparation légèrement moins nette entre les distributions spoof et bonafide, mais cela n'affecte pas son efficacité pratique.

---

### **Conclusion générale du Step 3**

Les deux modèles montrent une excellente capacité de généralisation sur des données totalement inédites :
On voit que le Random Forest apparaît comme le modèle le plus robuste opérationnellement  
  (meilleure accuracy, meilleur F1-score, moins d’erreurs sur les bonafide).
Le SVM reste très compétitif et affiche la meilleure ROC-AUC, signe d’une bonne séparation théorique entre classes.

Ces performances démontrent que notre approche basée sur des MFCC (26 dimensions) combinée aux modèles vus en cours, peut constituer une base solide pour la détection de deepfakes audio.
